In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ExpenseTransactionAnalysisWeek4") \
    .getOrCreate()

In [2]:
expenses = spark.read.csv("/content/expense_data_cleaned.csv", header=True, inferSchema=True)

In [3]:
user_details = spark.createDataFrame([
    (1, "arjun.mehta@email.com", "2025-01-15"),
    (2, "priya.nair@email.com", "2025-02-10"),
    (3, "karthik.iyer@email.com", "2025-01-22"),
    (4, "sneha.reddy@email.com", "2025-03-05"),
    (5, "rohan.verma@email.com", "2025-02-28"),
], ["user_id", "email", "signup_date"])

monthly_budgets = spark.createDataFrame([
    (1, 12000.00),
    (2, 10000.00),
    (3, 11000.00),
    (4, 9000.00),
    (5, 13000.00),
], ["user_id", "monthly_budget"])

In [7]:
combined = expenses.join(user_details, on="user_id")

combined.show(truncate=False)

+-------+----------+-----------+--------------+------------------+--------------+-------------------+---------------------+-----------+
|user_id|date      |user_name  |category      |amount            |payment_method|month              |email                |signup_date|
+-------+----------+-----------+--------------+------------------+--------------+-------------------+---------------------+-----------+
|1      |2026-05-08|Arjun Mehta|Groceries     |2119.36           |Net Banking   |2026-05-01 00:00:00|arjun.mehta@email.com|2025-01-15 |
|1      |2026-06-03|Arjun Mehta|Dining Out    |1223.54           |Card          |2026-06-01 00:00:00|arjun.mehta@email.com|2025-01-15 |
|1      |2026-05-06|Arjun Mehta|Shopping      |2434.47           |Net Banking   |2026-05-01 00:00:00|arjun.mehta@email.com|2025-01-15 |
|1      |2026-06-11|Arjun Mehta|Healthcare    |897.05            |Net Banking   |2026-06-01 00:00:00|arjun.mehta@email.com|2025-01-15 |
|1      |2026-05-27|Arjun Mehta|Dining Out    |1

In [6]:
monthly_summary = combined.groupBy("month", "user_id", "user_name", "email") \
    .agg(F.round(F.sum("amount"), 2).alias("total_spend"))

monthly_report = monthly_summary.join(monthly_budgets, on="user_id") \
    .withColumn("savings", F.round(F.col("monthly_budget") - F.col("total_spend"), 2)) \
    .withColumn(
        "alert_status",
        F.when(F.col("total_spend") > F.col("monthly_budget"), "Over Budget")
        .when(F.col("total_spend") < F.col("monthly_budget") * 0.7, "High Savings")
        .otherwise("On Track")
    ) \
    .select("month", "user_id", "user_name", "email", "total_spend", "monthly_budget", "savings", "alert_status") \
    .orderBy("month", "user_name")

monthly_report.show(truncate=False)

+-------------------+-------+------------+----------------------+-----------+--------------+--------+------------+
|month              |user_id|user_name   |email                 |total_spend|monthly_budget|savings |alert_status|
+-------------------+-------+------------+----------------------+-----------+--------------+--------+------------+
|2026-05-01 00:00:00|1      |Arjun Mehta |arjun.mehta@email.com |13619.49   |12000.0       |-1619.49|Over Budget |
|2026-05-01 00:00:00|3      |Karthik Iyer|karthik.iyer@email.com|9537.33    |11000.0       |1462.67 |On Track    |
|2026-05-01 00:00:00|2      |Priya Nair  |priya.nair@email.com  |10086.24   |10000.0       |-86.24  |Over Budget |
|2026-05-01 00:00:00|5      |Rohan Verma |rohan.verma@email.com |14437.11   |13000.0       |-1437.11|Over Budget |
|2026-05-01 00:00:00|4      |Sneha Reddy |sneha.reddy@email.com |10138.58   |9000.0        |-1138.58|Over Budget |
|2026-06-01 00:00:00|1      |Arjun Mehta |arjun.mehta@email.com |8846.66    |120

In [8]:
category_summary = combined.groupBy("month", "user_name", "category") \
    .agg(F.round(F.sum("amount"), 2).alias("category_spend")) \
    .orderBy("month", "user_name", F.col("category_spend").desc())

category_summary.show(truncate=False)

+-------------------+------------+--------------+--------------+
|month              |user_name   |category      |category_spend|
+-------------------+------------+--------------+--------------+
|2026-05-01 00:00:00|Arjun Mehta |Groceries     |6243.92       |
|2026-05-01 00:00:00|Arjun Mehta |Shopping      |2434.47       |
|2026-05-01 00:00:00|Arjun Mehta |Utilities     |1594.44       |
|2026-05-01 00:00:00|Arjun Mehta |Dining Out    |1274.41       |
|2026-05-01 00:00:00|Arjun Mehta |Entertainment |1123.58       |
|2026-05-01 00:00:00|Arjun Mehta |Transportation|948.68        |
|2026-05-01 00:00:00|Karthik Iyer|Shopping      |2782.36       |
|2026-05-01 00:00:00|Karthik Iyer|Dining Out    |2212.26       |
|2026-05-01 00:00:00|Karthik Iyer|Utilities     |1831.25       |
|2026-05-01 00:00:00|Karthik Iyer|Entertainment |1380.89       |
|2026-05-01 00:00:00|Karthik Iyer|Healthcare    |897.05        |
|2026-05-01 00:00:00|Karthik Iyer|Transportation|433.52        |
|2026-05-01 00:00:00|Priy

In [9]:
alerts_only = monthly_report.filter(F.col("alert_status") == "Over Budget").orderBy(F.col("total_spend").desc())

alerts_only.show(truncate=False)

+-------------------+-------+------------+----------------------+-----------+--------------+--------+------------+
|month              |user_id|user_name   |email                 |total_spend|monthly_budget|savings |alert_status|
+-------------------+-------+------------+----------------------+-----------+--------------+--------+------------+
|2026-06-01 00:00:00|3      |Karthik Iyer|karthik.iyer@email.com|17540.08   |11000.0       |-6540.08|Over Budget |
|2026-05-01 00:00:00|5      |Rohan Verma |rohan.verma@email.com |14437.11   |13000.0       |-1437.11|Over Budget |
|2026-05-01 00:00:00|1      |Arjun Mehta |arjun.mehta@email.com |13619.49   |12000.0       |-1619.49|Over Budget |
|2026-05-01 00:00:00|4      |Sneha Reddy |sneha.reddy@email.com |10138.58   |9000.0        |-1138.58|Over Budget |
|2026-05-01 00:00:00|2      |Priya Nair  |priya.nair@email.com  |10086.24   |10000.0       |-86.24  |Over Budget |
+-------------------+-------+------------+----------------------+-----------+---

In [ ]:
# monthly_report.write.format("delta").mode("overwrite").saveAsTable("expense_tracker.monthly_report")
# category_summary.write.format("delta").mode("overwrite").saveAsTable("expense_tracker.category_summary")
# alerts_only.write.format("delta").mode("overwrite").saveAsTable("expense_tracker.budget_alerts")

In [17]:
monthly_report.toPandas().to_csv("/content/monthly_report.csv",index= False)
category_summary.toPandas().to_csv("/content/category_summary.csv",index = False)
alerts_only.toPandas().to_csv("/content/budget_alerts.csv", index = False)

In [13]:
monthly_report.createOrReplaceTempView("monthly_report_view")

In [14]:
spark.sql("""
SELECT user_name, COUNT(*) AS months_over_budget, ROUND(SUM(savings), 2) AS total_overspend
FROM monthly_report_view
WHERE alert_status = 'Over Budget'
GROUP BY user_name
ORDER BY total_overspend ASC
""").show(truncate=False)

+------------+------------------+---------------+
|user_name   |months_over_budget|total_overspend|
+------------+------------------+---------------+
|Karthik Iyer|1                 |-6540.08       |
|Arjun Mehta |1                 |-1619.49       |
|Rohan Verma |1                 |-1437.11       |
|Sneha Reddy |1                 |-1138.58       |
|Priya Nair  |1                 |-86.24         |
+------------+------------------+---------------+



In [15]:
spark.sql("""
SELECT month, ROUND(AVG(total_spend), 2) AS avg_spend_across_users
FROM monthly_report_view
GROUP BY month
ORDER BY month
""").show(truncate=False)

+-------------------+----------------------+
|month              |avg_spend_across_users|
+-------------------+----------------------+
|2026-05-01 00:00:00|11563.75              |
|2026-06-01 00:00:00|10463.87              |
+-------------------+----------------------+

